In [102]:
import pandas as pd
from Levenshtein import distance
import numpy as np
import re
import unicodedata

In [103]:
#read the Compensation Office csv file into a dataframe
df_csv = pd.read_excel('/Users/mva/Desktop/Post-processing/2025_06_03_Liste der Entschädigungsämter mit Adressen, Varianten, Normalisierungen und Archiven.xlsx')
df_csv.sample(20)

,Layout class,Filename,Bundesland,Transkription Compensation Office,Normalisierung Compensation Office1,Normalisierung Compensation Office2,Zuständiges Archiv,Zuständiges Archiv - GND-Nummer,Unnamed: 8
28,BW-Hauptphase,1880_00_00_243_0,Baden-Württemberg,Landesamt für die Wiedergutmachung \nStuttgart...,Landesamt für die Wiedergutmachung Stuttgart,NaN,Landesarchiv Baden-Württemberg,10107080-9,NaN
230,offizielle Bezeichnungen und Adressen,NaN,Rheinland-Pfalz,Neustadt a.d. Weinstraße,NaN,NaN,Amt für Wiedergutmachung Saarburg,1354917804,NaN
54,BY-Spätphase,1924_01_12_84_0,Bayern,"Landesentschädigungsamt\nPrinz-Ludwig-Str. 5, ...",Landesentschädigungsamt München,NaN,Bayerisches Hauptstaatsarchiv,2005486-5,NaN
271,offizielle Bezeichnungen und Adressen,NaN,Niedersachsen,Der Präsident des Verwaltungsbezirks in Oldenb...,NaN,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN
120,HH-NI-NRW-SH-Hauptphase (abweichend 2),1927_05_06_17_0,Niedersachsen,Reg.-Präsident Hildesheim,Regierungspräsident Hildesheim,NaN,Niedersächsisches Landesarchiv,6510041-4,NaN
52,BY-Spätphase,1908_01_15_69_0,Bayern,Bayerisches\nLandesentschädigungsamt\nPrinz-Lu...,Bayerisches Landesentschädigungsamt München,NaN,Bayerisches Hauptstaatsarchiv,2005486-5,NaN
378,RLP-Frühe-Phase,1903_02_04_83_0,Rheinland-Pfalz,Regierungsbezirksamt\nfür Wiedergutmachung\nun...,Regierungsbezirksamt für Wiedergutmachung und ...,NaN,Landesarchivverwaltung Rheinland-Pfalz,2029741-5,NaN
216,offizielle Bezeichnungen und Adressen,NaN,Rheinland-Pfalz,Bezirksamt für Wiedergutmachung Trier,NaN,NaN,Amt für Wiedergutmachung Saarburg,1354917804,NaN
422,Tabellen-Typ (abweichend),1909_01_22_88_0,Bund,Oberfinanzdirektion: Freiburg,Oberfinanzdirektion Freiburg,NaN,Bundesarchiv (Koblenz),39454-3,Landesarchivverwaltung Rheinland-Pfalz
35,BW-Hauptphase,1876_01_04_2_0,Baden-Württemberg,Landesamt für die Wiedergutmachung\nStuttgart ...,Landesamt für die Wiedergutmachung Stuttgart,NaN,Landesarchiv Baden-Württemberg,10107080-9,NaN


In [104]:
#creaae a list of unique compensation offices from the csv file
gt = df_csv['Transkription Compensation Office '].unique().tolist()
print("Number of unique compensation offices: ", len(gt))

Number of unique compensation offices:  428


In [ ]:
#read the sliced extracted data jsonl file into a dataframe
df_json = pd.read_json('compoff_extracted.jsonl', lines=True)
df_json.sample(20)

,CompensationOffice1,BZKNr,filename
412139,Entschädigungsamt Berlin,374 179,1912_02_25_62_0.jpg
1458848,Bezirksamt für Wiedergutmachung KOBLENZ,448425,1907_08_27_99_0.jpg
339879,Stuttgart,122946/VII/37913,1934_07_19_13_0.jpg
1233523,Hess. Staatsministerium,16713,1892_05_11_38_0.jpg
1299202,Köln,19079a,1896_10_28_84_0.jpg
73446,Entschädigungsbehörde Köln,Art. V-56-II-Nr.: 66 508,1916_07_14_18_0.jpg
171729,Bezirksamt für Wiedergutmachung Koblenz,156968,1922_02_12_7_0.jpg
554391,Bezirksamt für Wiedergutmachung Koblenz,319125,1928_02_22_79_0.jpg
1698451,Köln,609 429,1919_12_08_17_0.jpg
1779047,NaN,BEG 38652,1902_04_15_63_0.jpg


In [105]:
#create a dataframe with the entries that have a non-null value in the CompensationOffice1 column
print("Number of cards: ",len(df_json))
mask_not_null = df_json['CompensationOffice1'].notna()
print(f'Number of non-null entries: {mask_not_null.sum()}')


Number of cards:  1901284
Number of non-null entries: 1825946


In [106]:
#create a function to normalise the compensation office names by removing punctuation and whitespace, converting to lowercase and replacing umlauts with their equivalent

#def normalise(word):
#    return re.sub(r'\s+', ' ', str(word)).strip().lower()
def normalize(s):
    if not isinstance(s, str):
        return ""
    
    s = s.lower()

    # Normalize umlauts and ß
    replacements = {
        "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"
    }
    for k, v in replacements.items():
        s = s.replace(k, v)
    
    # Normalize unicode (just in case)
    s = unicodedata.normalize("NFKD", s)
    
    # Remove punctuation and whitespace
    s = re.sub(r'[\W_]+', '', s)
        
    return s

gt_normalized = [normalize(office) for office in gt]

In [107]:
#create a mask for the entries in the json dataframe that match any of the normalized compensation office names in the gt list
mask_matched_0 = df_json['CompensationOffice1'].apply(normalize).isin(gt_normalized)
print(f'Number of matches: {mask_matched_0.sum()}')

df_unmatched = df_json[mask_not_null & ~mask_matched_0]
print(f'Number of unmatched entries: {len(df_unmatched)}')


Number of matches: 1494694
Number of unmatched entries: 331252


In [ ]:
#print the unique values in the CompensationOffice1 column of the unmatched dataframe and their counts
print(len(df_unmatched['CompensationOffice1'].unique().tolist()))
df_unmatched['CompensationOffice1'].value_counts()

27629


CompensationOffice1
Bezirksamt für Wiedergutmachung Köln                                   32785
Hess. Staatsministerium                                                29978
Freie und Hansestadt Hamburg Sozialbehörde Amt für Wiedergutmachung    16429
Landesamt für die Wiedergutmachung                                     14641
Hansestadt Hamburg, Sozialbehörde, Amt für Wiedergutmachung            11836
                                                                       ...  
Regierung Aachen (Welt.)                                                   1
Regierungspräsident Arasberg                                               1
Kreisssonderhilfsausschuss des Landkreises Garsch. Schlaumburg             1
Bayerisches Landesamt für den Regierungsbezirk München                     1
Reg.-Präsident /Präsident des Vereinsbezirks Hannover                      1
Name: count, Length: 27629, dtype: int64

In [191]:
#create a list of manually corrected compensation office names that are not in the gt list but are in the json dataframe and normalise them
list_corrections = ["Bezirksamt für Wiedergutmachung Köln", "Hess. Staatsministerium", "Freie und Hansestadt Hamburg Sozialbehörde Amt für Wiedergutmachung", "Hansestadt Hamburg, Sozialbehörde, Amt für Wiedergutmachung", 
                    "Der Innenminister des Landes Nordrhein-Westfalen", "Landesamt für die Wiedergutmachung Baden-Württemberg", "Oberfinanzdirektion Niedersachsen", "Statistisches Landesamt Nordrhein-Westfalen" , "Oberfinanzdirektion Niedersachsen"
                    , "Rhein. Pfalz - Berlin", "Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen Köln", "Aachen", "Landesbeirat für die Wiedergutmachung, Stuttgart", "Freiburg i.Br.", "Hamburg, Sozialbehörde, Amt für Wiedergutmachung",
                    "Freie und Hansestadt Hamburg", "Regierungsbezirk Mainz", "Hessisches Ministerium für Jugend, Familie und Gesundheit" , "Reg.-Präsident / Präsident des Verw.-Bezirks Hannover", "Amt für Wiedergutmachung des Landes Rheinland/Pfalz in Berlin Hauptabteilung II",
                    "Schleswig-Holstein in Kiel", "Kreissozialamt Hannover", "Reg.-Präsident Hannover", "Amt für Wiedergutmachung Rheinland-Pfalz" , "Saarburg Bezirksamt für Wiedergutmachung", "Freie und Hansestadt Hamburg, Arbeits- und Sozialbehörde, Amt für Wiedergutmachung",
                    "Niedersachsen-Präsident / Präsident des Verwaltungsbezirks Hannover", "Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen Neustadt a. d. Weinstraße", "KSHA Hannover", "Regierungsbezirk für Wiedergutmachung und verwaltete Vermögen Trier",
                    "Oberfinanzdirektion Magdeburg", "Regierungsbezirkskammer für Wiedergutmachung und verwaltete Vermögen Mainz","Regierungsamt Arnsberg (Westf.)", "Regierungsbezirkskammer für Wiedergutmachung und verwaltete Vermögen Mainz",
                    "Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen München", "Landesamt für die Wiedergutmachung Baden-Württemberg Stuttgart"]

list_corrections_normalized = [normalize(office) for office in list_corrections]
gt_completed = gt_normalized + list_corrections_normalized
df_json['CompensationOffice1'] = df_json['CompensationOffice1'].str.replace("Bayrische", "Bayerische")
mask_matched_1 = df_json['CompensationOffice1'].apply(normalize).isin(gt_completed)
print(f'Number of matches: {mask_matched_1.sum()}')
df_unmatched = df_json[mask_not_null & ~mask_matched_1]
print(f'Number of unmatched entries: {len(df_unmatched)}')
print(f"Unique unmatched entries: {len(df_unmatched['CompensationOffice1'].unique().tolist())}")
percentage = len(df_unmatched) / len(mask_not_null) * 100
print(f"Percentage of unmatched entries: {percentage:.2f}%")



Number of matches: 1628175
Number of unmatched entries: 197771
Unique unmatched entries: 27325
Percentage of unmatched entries: 10.40%


In [192]:
df_unmatched['CompensationOffice1'].value_counts().head(30)

CompensationOffice1
Landesamt für die Wiedergutmachung                                       14641
Bayerische Kartei                                                         9925
Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen         7261
Bundeskartei                                                              4884
Bezirksamt für Wiedergutmachung                                           4130
HNG                                                                       3402
Regierungsbezirksamt für Wiedergutmachung und kontrollierte Vermögen      3354
Bayerisches Landesarchiv                                                  2585
Regierungsbezirks-Amt für Wiedergutmachung und kontrollierte Vermögen     2557
Bezirksamt für Wiedergutmachung 65 Mainz                                  1304
Bayerisches Landesamt für Verfolgtenfragen                                1291
Kreissonderhilfsausschuss                                                 1101
Barburg                         

# Matching based on Levenshtein Distance

In [193]:
#  return the minimum distance and the best match up to a threshold of half the length of the cleaned name
def get_min_distance_dynamic_normalized(name):
    name_clean = normalize(name)
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in gt_completed:
        office_clean = normalize(office)
        d = distance(name_clean, office_clean)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_unmatched['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

lev_matches = pd.DataFrame({
    'CompensationOffice1': df_unmatched['CompensationOffice1'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})


In [194]:
#print the matches with a distance of 1 and 2
df_matches_2 = lev_matches[lev_matches['MatchDistance'].isin([1, 2])]
df_matches_2.sample(25)

,CompensationOffice1,MatchedOffice,MatchDistance
1005098,Am. für Wiedergutmachung und kontrollierte Ver...,amtfuerwiedergutmachungundkontrolliertevermoeg...,1.0
551242,LR8 NW,lrbnw,1.0
750459,Thür,trier,2.0
1150474,R.A. Pfalz-Berlin,rhpfalzberlin,1.0
569787,Sassburg,saarburg,2.0
800643,Regierungspräsident Arnberg,regierungspraesidentarnsberg,1.0
1620622,Landesamt für die Wiedergutmachung Stuttgart N...,landesamtfuerdiewiedergutmachungstuttgartneuew...,1.0
1616184,HöIn,koeln,2.0
314510,Hamburg I,hamburg,1.0
552476,Holn,koeln,2.0


In [195]:
#print the counts of the match distances of 1 and 2
df_matches_2['MatchDistance'].value_counts()
print(f"Number of matches with distance 1 or 2: {len(df_matches_2)}")
print(f"Number of unmatched entries after adding the close matches: {len(df_unmatched) - len(df_matches_2)}")

Number of matches with distance 1 or 2: 27826
Number of unmatched entries after adding the close matches: 169945


In [201]:
#remove the matches with distance 1 and 2 from the unmatched dataframe
df_unmatched_filtered = df_unmatched[~df_unmatched['CompensationOffice1'].isin(lev_matches[lev_matches['MatchDistance'].isin([1, 2])]['CompensationOffice1'])]
df_unmatched_filtered['CompensationOffice1'].value_counts().iloc[30:40]

CompensationOffice1
Amt für Wiedergutmachung des Landes Rheinland-Pfalz                470
Regierung Arnsberg (Wf.)                                           459
Doppelmeldung                                                      449
Amt für Wiedergutmachung Stadt Bonn                                447
Bundesamt für Wiedergutmachung                                     439
Niedersachsen-Präsident / Präsident des Verwaltungsgerichtshofs    437
Regierung Köln                                                     421
Bundesarchiv                                                       408
Reg.-Präsidium Hildesheim                                          396
Entscheidungsbehörde Köln                                          396
Name: count, dtype: int64

In [202]:
df_matches_correctable = lev_matches.dropna()
print(len(df_matches_correctable))
df_matches_impossible = lev_matches[lev_matches['MatchedOffice'].isna()]
impossible_list = df_matches_impossible['CompensationOffice1'].tolist()
print(len(impossible_list), len(set(impossible_list)))
# print the unique values in the impossible list and their counts in descending order
for office in sorted(set(impossible_list), key=impossible_list.count, reverse=True):
    count = impossible_list.count(office)
    print(f"{office}: {count}")



145672
52099 7005
Bayerische Kartei: 9925
Bundeskartei: 4884
HNG: 3402
Bayerisches Landesarchiv: 2585
Kreissozialamt: 958
Bundesamt: 834
KSHA: 750
Kreissozialhilfsausschuss: 712
Köln, Riehler Pl.2, 5000 Köln 1: 477
Doppelmeldung: 449
Bundesarchiv: 408
Erfurt: 386
An Bundeskartei: 371
BHF: 346
BEG: 340
Bayerisches Hauptstaatsarchiv: 321
Magistrat der Stadt Bremen: 306
Montabaur: 291
BMF: 269
K81: 268
K81n: 267
Frankfurt/M.: 267
Ansbach: 257
Sondervermögens-u. Bauverwaltung: 242
Bayerisches Landesamt: 241
Aurich/Ostfriesland: 235
Magistrat der Stadt Bremenhaven: 224
VB3: 218
Bayerisches Verfolgtenamt: 209
Landeszentrale für die Vergütung von Opfern des Nationalsozialismus: 206
KSH: 205
Köln, Riehler Pl. 2, 5000 Köln 1: 204
Watenstedt-Salzgitter: 185
Bayerisches Landeszentrum für die Opfer des Nationalsozialismus: 181
Pirmasens: 181
Kaiserslautern: 178
Bayerisches Landesamt für die Opfer des Nationalsozialismus: 173
Höherer Kommissar der Vereinten Nationen für Flüchtlinge: 166
Kreissonder

In [203]:
print(f"Max MatchDistance: {lev_matches['MatchDistance'].max()}")

Max MatchDistance: 58.0


In [205]:
total = len(lev_matches)-len(df_matches_impossible)
max_dist = int(lev_matches['MatchDistance'].max()) + 1
for i in range(1, max_dist):
    count = (lev_matches['MatchDistance'] == i).sum()
    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 1: 17610 (12.09%)
MatchDistance = 2: 10216 (7.01%)
MatchDistance = 3: 9283 (6.37%)
MatchDistance = 4: 7249 (4.98%)
MatchDistance = 5: 25033 (17.18%)
MatchDistance = 6: 7157 (4.91%)
MatchDistance = 7: 5696 (3.91%)
MatchDistance = 8: 23344 (16.03%)
MatchDistance = 9: 5215 (3.58%)
MatchDistance = 10: 3719 (2.55%)
MatchDistance = 11: 5334 (3.66%)
MatchDistance = 12: 2539 (1.74%)
MatchDistance = 13: 3634 (2.49%)
MatchDistance = 14: 3022 (2.07%)
MatchDistance = 15: 2023 (1.39%)
MatchDistance = 16: 2193 (1.51%)
MatchDistance = 17: 2679 (1.84%)
MatchDistance = 18: 2706 (1.86%)
MatchDistance = 19: 1229 (0.84%)
MatchDistance = 20: 1014 (0.70%)
MatchDistance = 21: 1700 (1.17%)
MatchDistance = 22: 362 (0.25%)
MatchDistance = 23: 375 (0.26%)
MatchDistance = 24: 360 (0.25%)
MatchDistance = 25: 477 (0.33%)
MatchDistance = 26: 168 (0.12%)
MatchDistance = 27: 105 (0.07%)
MatchDistance = 28: 294 (0.20%)
MatchDistance = 29: 202 (0.14%)
MatchDistance = 30: 113 (0.08%)
MatchDistance = 31: 5

In [209]:
df_nomatches = lev_matches[(lev_matches['MatchDistance'] > 0) | (lev_matches['MatchDistance'].isna())]

df_nomatches.sort_values('MatchDistance', ascending=True).to_excel('compensation_office_nomatches.xlsx', index=False)